# 03 Prepare Training Data

### Check current BioRED train, test, dev datasets

In [1]:
import csv
from pathlib import Path
import pandas as pd

import numpy as np

In [3]:
# Count the number of unique pmids for each of the train/dev/test files in the BioRED dataset.
# Count also the number of rows for each file and the average number of rows per unique pmid.

# Define path
data_dir = Path(r"D:\Users\Jessica\kg-construction\data\00_raw\biorex\ncbi_relation\processed")
files = ["train.tsv", "dev.tsv", "test.tsv"]

stats = []

for file in files:
    file_path = data_dir / file
    
    # Read only column 0 (PMID) for high performance
    df = pd.read_csv(
        file_path, 
        sep="\t", 
        header=None, 
        usecols=[0], 
        names=["pmid"], 
        quoting=3
    )
    
    # Count frequency of each PMID
    pmid_counts = df["pmid"].value_counts()
    
    total_rows = len(df)
    unique_pmids = len(pmid_counts)
    avg_rows = pmid_counts.mean()
    
    stats.append({
        "File": file,
        "Unique PMIDs": f"{unique_pmids:,}",
        "Total Rows": f"{total_rows:,}",
        "Avg Rows / PMID": f"{avg_rows:.2f}",
        "Median Rows": int(pmid_counts.median()),
        "Max Rows": int(pmid_counts.max())
    })

# Display summary table
summary_df = pd.DataFrame(stats)
print(summary_df.to_string(index=False))

     File Unique PMIDs Total Rows Avg Rows / PMID  Median Rows  Max Rows
train.tsv          398     22,896           57.53           27       855
  dev.tsv           98      6,659           67.95           39       462
 test.tsv          100      7,591           75.91           34       560


In [4]:
# Load the duplicates train, test, and dev files into DataFrames for further analysis.

# Correct 10-column schema
columns = [
    "pmid",
    "Entity_1_Type",
    "Entity_2_Type",
    "Entity_1_ID",
    "Entity_2_ID",
    "is_in_same_sent", # Whether the two entities are in the same sentence
    "min_sents_window", # 0 = same sentence
    "sentence",
    "extra",
    "Relation_Type"    
]

# Read files into separate variables
train_df = pd.read_csv(data_dir / "train.tsv", sep="\t", header=None, names=columns, quoting=csv.QUOTE_NONE)
dev_df   = pd.read_csv(data_dir / "dev.tsv",   sep="\t", header=None, names=columns, quoting=csv.QUOTE_NONE)
test_df  = pd.read_csv(data_dir / "test.tsv",  sep="\t", header=None, names=columns, quoting=csv.QUOTE_NONE)

# Quick verification
print(f"train_df loaded: {train_df.shape}")
print(f"unique pmids in train_df: {train_df['pmid'].nunique()}")
print(f"dev_df   loaded: {dev_df.shape}")
print(f"unique pmids in dev_df: {dev_df['pmid'].nunique()}")
print(f"test_df  loaded: {test_df.shape}")
print(f"unique pmids in test_df: {test_df['pmid'].nunique()}")

# Verification check
train_df

train_df loaded: (22896, 10)
unique pmids in train_df: 398
dev_df   loaded: (6659, 10)
unique pmids in dev_df: 98
test_df  loaded: (7591, 10)
unique pmids in test_df: 100


,pmid,Entity_1_Type,Entity_2_Type,Entity_1_ID,Entity_2_ID,is_in_same_sent,min_sents_window,sentence,extra,Relation_Type
0,10491763,GeneOrGeneProduct,GeneOrGeneProduct,6927,rs74805019,True,0,What is [Litcoin] between @GeneOrGeneProductSr...,NaN,NaN
1,10491763,GeneOrGeneProduct,GeneOrGeneProduct,3630,rs74805019,True,0,What is [Litcoin] between @GeneOrGeneProductSr...,NaN,NaN
2,10491763,ChemicalEntity,GeneOrGeneProduct,D005947,6927,True,0,What is [Litcoin] between @ChemicalEntitySrc$ ...,NaN,NaN
3,10491763,GeneOrGeneProduct,GeneOrGeneProduct,3651,rs74805019,True,0,What is [Litcoin] between @GeneOrGeneProductSr...,NaN,NaN
4,10491763,ChemicalEntity,GeneOrGeneProduct,D005947,3172,True,0,What is [Litcoin] between @ChemicalEntitySrc$ ...,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
22891,24927617,ChemicalEntity,DiseaseOrPhenotypicFeature,D019821,D012206,True,0,What is [Litcoin] between @ChemicalEntitySrc$ ...,NaN,Positive_Correlation
22892,24927617,ChemicalEntity,ChemicalEntity,D012254,C417083,True,0,What is [Litcoin] between @ChemicalEntitySrc$ ...,NaN,Cotreatment
22893,24927617,ChemicalEntity,ChemicalEntity,D019821,D000998,True,0,What is [Litcoin] between @ChemicalEntitySrc$ ...,NaN,Drug_Interaction
22894,24927617,ChemicalEntity,ChemicalEntity,C486464,D012254,True,0,What is [Litcoin] between @ChemicalEntitySrc$ ...,NaN,Cotreatment


In [ ]:
print(train_df['is_in_same_sent'].value_counts())
print(train_df['min_sents_window'].value_counts())

is_in_same_sent
True     22530
False      366
Name: count, dtype: int64
min_sents_window
0      22530
1        198
100      168
Name: count, dtype: int64


In [7]:
# 1. Combine entity columns from all three DataFrames
combined = pd.concat([train_df, dev_df, test_df])[
    ['Entity_1_Type', 'Entity_2_Type']
].dropna()

# 2. Sort pairs alphabetically row-by-row to ignore order (e.g., ['B', 'A'] -> ['A', 'B'])
sorted_pairs = np.sort(combined.values, axis=1)

# 3. Extract unique pairs as a DataFrame
unique_pairs_df = (
    pd.DataFrame(sorted_pairs, columns=['Entity_A', 'Entity_B'])
    .drop_duplicates()
    .reset_index(drop=True)
)

print(unique_pairs_df)

            Entity_A           Entity_B
0  GeneOrGeneProduct  GeneOrGeneProduct
1     ChemicalEntity  GeneOrGeneProduct
2     ChemicalEntity            Disease
3     ChemicalEntity     ChemicalEntity
4            Disease  GeneOrGeneProduct


### Create Gemini train, test, dev datasets
* Duplicated the `ncbi_relation` folder and the files contained, renaming it to `ncbi_relation_gemini_annotations`. I will edit the files in this folder directly (leave the PubTator files for now and focus on the processed .tsv files)
* Used the BioREx conversion script (src/dataset_format_converter/utils.py, function dump_documents_2_bert_format) to map the columns 
    * The vast majority of entity pairs are in the same sentence
* BioREx does not predict novelty 

In [170]:
# Load in LLM generated triples

# Path to the LLM extraction CSV
csv_path = Path(r"D:\Users\Jessica\kg-construction\outputs\02_LLM_RE\full_triple_extraction_BioRED600_df.csv")

# Read into DataFrame (headers parsed automatically)
llm_re_df = pd.read_csv(csv_path)

# Ensure 'pmid' is stored as string for consistent matching across dataframes
llm_re_df["pmid"] = llm_re_df["pmid"].astype(str)

# Quick verification
print(f"llm_re_df shape: {llm_re_df.shape}")
print("\nColumns:")
print(llm_re_df.columns.tolist())
print('\nNumber of Unique PMIDS:', llm_re_df['pmid'].nunique())
llm_re_df

llm_re_df shape: (6121, 10)

Columns:
['pmid', 'Entity_1_ID', 'Entity_1_Name', 'Entity_1_Type', 'Relation_Type', 'Entity_2_ID', 'Entity_2_Name', 'Entity_2_Type', 'Novelty_Triage', 'Evidence_Quote']

Number of Unique PMIDS: 594


,pmid,Entity_1_ID,Entity_1_Name,Entity_1_Type,Relation_Type,Entity_2_ID,Entity_2_Name,Entity_2_Type,Novelty_Triage,Evidence_Quote
0,18340638,D001379,azathioprine,ChemicalEntity,Positive_Correlation,D000740,anemia,DiseaseOrPhenotypicFeature,No,The side effects of azathioprine include anemia
1,18340638,D000740,anemia,DiseaseOrPhenotypicFeature,Positive_Correlation,D010718,PS,ChemicalEntity,No,anemia could result from accelerated suicidal ...
2,18340638,D001379,azathioprine,ChemicalEntity,Positive_Correlation,D010718,PS,ChemicalEntity,Novel,erythrocytes from patients indeed showed a sig...
3,18340638,308,annexin V,GeneOrGeneProduct,Bind,D010718,PS,ChemicalEntity,No,"According to annexin V binding, erythrocytes f..."
4,18340638,C059715,Fluo3,ChemicalEntity,Association,D002118,Ca2+,ChemicalEntity,No,cytosolic Ca2+ activity (Fluo3 fluorescence)
...,...,...,...,...,...,...,...,...,...,...
6116,17042910,C065179,Ato,ChemicalEntity,Cotreatment,D003907,Dex,ChemicalEntity,Novel,Male SD rats (n = 30) were treated with Ato (5...
6117,17042910,D003907,Dex,ChemicalEntity,Positive_Correlation,D006973,hypertension,DiseaseOrPhenotypicFeature,Novel,"In rats treated with Dex alone, SBP was increa..."
6118,17042910,C065179,Ato,ChemicalEntity,Negative_Correlation,D006973,hypertension,DiseaseOrPhenotypicFeature,Novel,"In the Ato + Dex group, SBP was increased from..."
6119,17042910,C065179,Ato,ChemicalEntity,Positive_Correlation,24598,eNOS,GeneOrGeneProduct,Novel,eNOS mRNA expression were greater in the Dex +...


In [171]:
# Convert the pmid column across all DataFrames to string type for consistent matching0.......
for df in [train_df, dev_df, test_df, llm_re_df]:
    df["pmid"] = df["pmid"].astype(str)

In [172]:
len(llm_re_df["pmid"].unique())

594

In [173]:
# Across train_df, dev_df, and test_df keep only the rows where the pmid is present in llm_re_df. This ensures that we are only working with PMIDs that have corresponding LLM generated triples.
train_df = train_df[train_df["pmid"].isin(llm_re_df["pmid"])]
dev_df = dev_df[dev_df["pmid"].isin(llm_re_df["pmid"])]
test_df = test_df[test_df["pmid"].isin(llm_re_df["pmid"])]

# Print the number of unique PMIDs in each filtered DataFrame
print(f"Filtered train_df shape: {train_df.shape}, Unique PMIDs: {train_df['pmid'].nunique()}")
print(f"Filtered dev_df shape: {dev_df.shape}, Unique PMIDs: {dev_df['pmid'].nunique()}")
print(f"Filtered test_df shape: {test_df.shape}, Unique PMIDs: {test_df['pmid'].nunique()}")

Filtered train_df shape: (22864, 10), Unique PMIDs: 395
Filtered dev_df shape: (6659, 10), Unique PMIDs: 98
Filtered test_df shape: (7591, 10), Unique PMIDs: 100


In [174]:
import pandas as pd
import numpy as np

# Ensure PMIDs and Entity IDs are strings for clean matching
for df in [train_df, dev_df, test_df, llm_re_df]:
    df["pmid"] = df["pmid"].astype(str).str.strip()
    
    # Handle variations in ID column naming
    if "entity1_id" in df.columns:
        df["entity1_id"] = df["entity1_id"].astype(str).str.strip()
        df["entity2_id"] = df["entity2_id"].astype(str).str.strip()
    elif "Entity_1_ID" in df.columns:
        df["Entity_1_ID"] = df["Entity_1_ID"].astype(str).str.strip()
        df["Entity_2_ID"] = df["Entity_2_ID"].astype(str).str.strip()

# Build bidirectional lookup map: (pmid, id_a, id_b) -> Relation_Type
llm_relation_map = {}

for _, row in llm_re_df.iterrows():
    pmid = row["pmid"]
    e1_id = row["Entity_1_ID"]
    e2_id = row["Entity_2_ID"]
    rel = row["Relation_Type"]
    
    # Map forward order
    llm_relation_map[(pmid, e1_id, e2_id)] = rel
    # Map reverse order (swapped match)
    llm_relation_map[(pmid, e2_id, e1_id)] = rel

In [175]:
def update_relation_types(df, lookup_map):
    """
    Updates the relation_type column based on (pmid, entity1_id, entity2_id) lookup.
    Sets relation_type to NaN if pair is not found in lookup_map.
    """
    # Create tuple key for every row in the target dataframe
    keys = list(zip(df["pmid"], df["Entity_1_ID"], df["Entity_2_ID"]))
    
    # Look up each pair; returns None if not found
    updated_relations = [lookup_map.get(k, np.nan) for k in keys]
    
    # Assign back to relation_type column
    df["Relation_Type"] = updated_relations
    return df

# Create updated copies stored in separate variables
train_updated_df = update_relation_types(train_df.copy(), llm_relation_map)
dev_updated_df   = update_relation_types(dev_df.copy(),   llm_relation_map)
test_updated_df  = update_relation_types(test_df.copy(),  llm_relation_map)

In [176]:
test_updated_df

,pmid,Entity_1_Type,Entity_2_Type,Entity_1_ID,Entity_2_ID,is_in_same_sent,min_sents_window,sentence,extra,Relation_Type
0,15485686,GeneOrGeneProduct,GeneOrGeneProduct,c|SUB|G|CODON1763|A,p|SUB|V|1763|M,True,0,What is [Litcoin] between @GeneOrGeneProductSr...,NaN,NaN
1,15485686,ChemicalEntity,DiseaseOrPhenotypicFeature,D008801,D017180,True,0,What is [Litcoin] between @ChemicalEntitySrc$ ...,NaN,Negative_Correlation
2,15485686,GeneOrGeneProduct,GeneOrGeneProduct,6331,c|SUB|G|CODON1763|A,True,0,What is [Litcoin] between @GeneOrGeneProductSr...,NaN,NaN
3,15485686,ChemicalEntity,GeneOrGeneProduct,D008801,p|SUB|V||M,True,0,What is [Litcoin] between @ChemicalEntitySrc$ ...,NaN,NaN
4,15485686,ChemicalEntity,GeneOrGeneProduct,D013779,p|SUB|V|1764|M,True,0,What is [Litcoin] between @ChemicalEntitySrc$ ...,NaN,Negative_Correlation
...,...,...,...,...,...,...,...,...,...,...
7586,24840785,DiseaseOrPhenotypicFeature,GeneOrGeneProduct,D066126,81649,True,0,What is [Litcoin] between @DiseaseOrPhenotypic...,NaN,NaN
7587,24840785,ChemicalEntity,ChemicalEntity,D000157,D012964,True,0,What is [Litcoin] between @ChemicalEntitySrc$ ...,NaN,NaN
7588,24840785,GeneOrGeneProduct,GeneOrGeneProduct,81649,24533,True,0,What is [Litcoin] between @GeneOrGeneProductSr...,NaN,NaN
7589,24840785,ChemicalEntity,DiseaseOrPhenotypicFeature,D012964,D064420,True,0,What is [Litcoin] between @ChemicalEntitySrc$ ...,NaN,NaN


In [177]:
for name, df in [("train_df", train_updated_df), ("dev_df", dev_updated_df), ("test_df", test_updated_df)]:
    total_rows = len(df)
    matched_rows = df["Relation_Type"].notna().sum()
    nan_rows = df["Relation_Type"].isna().sum()
    
    print(f"[{name}]")
    print(f"  Total Rows   : {total_rows:,}")
    print(f"  Matched (LLM): {matched_rows:,} ({matched_rows/total_rows:.1%})")
    print(f"  Unmatched    : {nan_rows:,}\n")

[train_df]
  Total Rows   : 22,864
  Matched (LLM): 3,379 (14.8%)
  Unmatched    : 19,485

[dev_df]
  Total Rows   : 6,659
  Matched (LLM): 1,059 (15.9%)
  Unmatched    : 5,600

[test_df]
  Total Rows   : 7,591
  Matched (LLM): 1,041 (13.7%)
  Unmatched    : 6,550



In [178]:
llm_re_df[llm_re_df['pmid'] == '24840785']

,pmid,Entity_1_ID,Entity_1_Name,Entity_1_Type,Relation_Type,Entity_2_ID,Entity_2_Name,Entity_2_Type,Novelty_Triage,Evidence_Quote
1124,24840785,D000157,aconitine,ChemicalEntity,Positive_Correlation,D002118,Ca(2+),ChemicalEntity,Novel,aconitine significantly aggravates Ca(2+) over...
1125,24840785,D002118,Ca(2+),ChemicalEntity,Positive_Correlation,D001145,arrhythmia,DiseaseOrPhenotypicFeature,Novel,Ca(2+) overload lead to accelerated beating rh...
1126,24840785,D000157,aconitine,ChemicalEntity,Positive_Correlation,81649,P38,GeneOrGeneProduct,Novel,promotes apoptotic development via phosphoryla...
1127,24840785,D000157,aconitine,ChemicalEntity,Positive_Correlation,D066126,cardiotoxicity,DiseaseOrPhenotypicFeature,No,cardiotoxicity of aconitine
1128,24840785,D012964,Na(+),ChemicalEntity,Association,D066126,cardiotoxicity,DiseaseOrPhenotypicFeature,No,voltage-dependent Na(+) channels have pivotal ...
1129,24840785,D000157,aconitine,ChemicalEntity,Positive_Correlation,D011041,poisoning,DiseaseOrPhenotypicFeature,No,aconitine poisoning
1130,24840785,D000157,aconitine,ChemicalEntity,Positive_Correlation,D009202,myocardial injury,DiseaseOrPhenotypicFeature,Novel,aconitine resulted in myocardial injury
1131,24840785,D000157,aconitine,ChemicalEntity,Positive_Correlation,D064420,cytotoxicity,DiseaseOrPhenotypicFeature,Novel,reduced NRVMs viability dose-dependently.
1132,24840785,D000157,aconitine,ChemicalEntity,Negative_Correlation,24224,BCL-2,GeneOrGeneProduct,Novel,anti-apoptotic protein BCL-2 expression was do...


In [179]:
# Visually assess the differences between the original and updated relation types for a specific PMID. This helps to verify that the updates are being applied correctly.
# target_pmid = "24927617"
target_pmid = "24840785"

# Merge original and updated DataFrames on PMID and entity pairs to see differences
comparison_df = pd.merge(
    # train_df[train_df["pmid"] == target_pmid][["pmid", "Entity_1_ID", "Entity_2_ID", "Relation_Type"]],
    # train_updated_df[train_updated_df["pmid"] == target_pmid][["Entity_1_ID", "Entity_2_ID", "Relation_Type"]],
    test_df[test_df["pmid"] == target_pmid][["pmid", "Entity_1_ID", "Entity_2_ID", "Relation_Type"]],
    test_updated_df[test_updated_df["pmid"] == target_pmid][["Entity_1_ID", "Entity_2_ID", "Relation_Type"]],
    on=["Entity_1_ID", "Entity_2_ID"],
    suffixes=("_original", "_llm_updated")
)

print(f"=== BEFORE vs. AFTER COMPARISON (PMID: {target_pmid}) ===")
print(comparison_df.to_string(index=False))

=== BEFORE vs. AFTER COMPARISON (PMID: 24840785) ===
    pmid Entity_1_ID Entity_2_ID Relation_Type_original Relation_Type_llm_updated
24840785     D000157     D002118   Positive_Correlation      Positive_Correlation
24840785     D002118     D066126                    NaN                       NaN
24840785     D000157     D001710                    NaN                       NaN
24840785     D002118     D001710                    NaN                       NaN
24840785     D009202       24533                    NaN                       NaN
24840785     D002118     D009202                    NaN                       NaN
24840785     D064420       24224                    NaN                       NaN
24840785     D002118     D064420                    NaN                       NaN
24840785     D064420       81649                    NaN                       NaN
24840785     D000157     D001145   Positive_Correlation                       NaN
24840785     D012964       24533             

In [161]:
test_updated_df

,pmid,Entity_1_Type,Entity_2_Type,Entity_1_ID,Entity_2_ID,is_in_same_sent,min_sents_window,sentence,extra,Relation_Type
0,15485686,GeneOrGeneProduct,GeneOrGeneProduct,c|SUB|G|CODON1763|A,p|SUB|V|1763|M,True,0,What is [Litcoin] between @GeneOrGeneProductSr...,NaN,NaN
1,15485686,ChemicalEntity,DiseaseOrPhenotypicFeature,D008801,D017180,True,0,What is [Litcoin] between @ChemicalEntitySrc$ ...,NaN,Negative_Correlation
2,15485686,GeneOrGeneProduct,GeneOrGeneProduct,6331,c|SUB|G|CODON1763|A,True,0,What is [Litcoin] between @GeneOrGeneProductSr...,NaN,NaN
3,15485686,ChemicalEntity,GeneOrGeneProduct,D008801,p|SUB|V||M,True,0,What is [Litcoin] between @ChemicalEntitySrc$ ...,NaN,NaN
4,15485686,ChemicalEntity,GeneOrGeneProduct,D013779,p|SUB|V|1764|M,True,0,What is [Litcoin] between @ChemicalEntitySrc$ ...,NaN,Negative_Correlation
...,...,...,...,...,...,...,...,...,...,...
7586,24840785,DiseaseOrPhenotypicFeature,GeneOrGeneProduct,D066126,81649,True,0,What is [Litcoin] between @DiseaseOrPhenotypic...,NaN,NaN
7587,24840785,ChemicalEntity,ChemicalEntity,D000157,D012964,True,0,What is [Litcoin] between @ChemicalEntitySrc$ ...,NaN,NaN
7588,24840785,GeneOrGeneProduct,GeneOrGeneProduct,81649,24533,True,0,What is [Litcoin] between @GeneOrGeneProductSr...,NaN,NaN
7589,24840785,ChemicalEntity,DiseaseOrPhenotypicFeature,D012964,D064420,True,0,What is [Litcoin] between @ChemicalEntitySrc$ ...,NaN,NaN


In [180]:
target_pmid = "18340638"

print(f"=== Ground Truth / Updated Split (PMID: {target_pmid}) ===")
for name, df in [
    ("train_df", train_df),
    ("dev_df", dev_df),
    ("test_df", test_df),
]:
    matches = df[df["pmid"] == target_pmid]
    if not matches.empty:
        print(f"Found PMID {target_pmid} in {name} ({len(matches)} rows)!")

print(f"\n=== LLM Extractions CSV (PMID: {target_pmid}) ===")
llm_rows = llm_re_df[llm_re_df["pmid"] == target_pmid][
    ["Entity_1_ID", "Entity_2_ID", "Relation_Type", "Entity_1_Name", "Entity_2_Name"]
]
print(llm_rows.to_string(index=False))

=== Ground Truth / Updated Split (PMID: 18340638) ===
Found PMID 18340638 in train_df (20 rows)!

=== LLM Extractions CSV (PMID: 18340638) ===
Entity_1_ID Entity_2_ID        Relation_Type Entity_1_Name Entity_2_Name
    D001379     D000740 Positive_Correlation  azathioprine        anemia
    D000740     D010718 Positive_Correlation        anemia            PS
    D001379     D010718 Positive_Correlation  azathioprine            PS
        308     D010718                 Bind     annexin V            PS
    C059715     D002118          Association         Fluo3          Ca2+
    D001379     D002118 Positive_Correlation  azathioprine          Ca2+
    D001379         308 Positive_Correlation  azathioprine     annexin V
    D002118     D001379 Positive_Correlation          Ca2+  azathioprine
    D001379     D000740 Positive_Correlation  azathioprine        anemia


In [181]:
train_updated_df

,pmid,Entity_1_Type,Entity_2_Type,Entity_1_ID,Entity_2_ID,is_in_same_sent,min_sents_window,sentence,extra,Relation_Type
0,10491763,GeneOrGeneProduct,GeneOrGeneProduct,6927,rs74805019,True,0,What is [Litcoin] between @GeneOrGeneProductSr...,NaN,NaN
1,10491763,GeneOrGeneProduct,GeneOrGeneProduct,3630,rs74805019,True,0,What is [Litcoin] between @GeneOrGeneProductSr...,NaN,NaN
2,10491763,ChemicalEntity,GeneOrGeneProduct,D005947,6927,True,0,What is [Litcoin] between @ChemicalEntitySrc$ ...,NaN,NaN
3,10491763,GeneOrGeneProduct,GeneOrGeneProduct,3651,rs74805019,True,0,What is [Litcoin] between @GeneOrGeneProductSr...,NaN,NaN
4,10491763,ChemicalEntity,GeneOrGeneProduct,D005947,3172,True,0,What is [Litcoin] between @ChemicalEntitySrc$ ...,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
22891,24927617,ChemicalEntity,DiseaseOrPhenotypicFeature,D019821,D012206,True,0,What is [Litcoin] between @ChemicalEntitySrc$ ...,NaN,Positive_Correlation
22892,24927617,ChemicalEntity,ChemicalEntity,D012254,C417083,True,0,What is [Litcoin] between @ChemicalEntitySrc$ ...,NaN,Cotreatment
22893,24927617,ChemicalEntity,ChemicalEntity,D019821,D000998,True,0,What is [Litcoin] between @ChemicalEntitySrc$ ...,NaN,NaN
22894,24927617,ChemicalEntity,ChemicalEntity,C486464,D012254,True,0,What is [Litcoin] between @ChemicalEntitySrc$ ...,NaN,Cotreatment


In [183]:
gemini_data_dir = Path(
    r"D:\Users\Jessica\kg-construction\data\00_raw\biorex\ncbi_relation_gemini_annotations\processed"
)

# List of updated DataFrames to process
updated_dfs = [
    ("train.tsv", train_updated_df),
    ("dev.tsv", dev_updated_df),
    ("test.tsv", test_updated_df),
]

for filename, df in updated_dfs:
    # 1. Fill NaN/missing values in Relation_Type with the string "None"
    df["Relation_Type"] = df["Relation_Type"].fillna("None")

    # 2. Ensure extra column contains empty strings so it renders as a blank tab
    df["extra"] = ""

    # 3. Ensure exact 10-column ordering
    expected_cols = [
        "pmid",
        "Entity_1_Type",
        "Entity_2_Type",
        "Entity_1_ID",
        "Entity_2_ID",
        "is_in_same_sent",
        "min_sents_window",
        "sentence",
        "extra",
        "Relation_Type",
    ]
    df_export = df[expected_cols]

    # 4. Save to TSV
    df_export.to_csv(
        gemini_data_dir / filename,
        sep="\t",
        index=False,
        header=False,
        quoting=csv.QUOTE_NONE,
    )

print("Export complete! Files match the target 10-column structure.")

Export complete! Files match the target 10-column structure.


In [184]:
gemini_data_dir = Path(r"D:\Users\Jessica\kg-construction\data\00_raw\biorex\ncbi_relation_gemini_annotations\processed")

# Write back to disk the updated DataFrames to TSV files, ensuring that the output format matches the expected schema for downstream processing.
train_updated_df.to_csv(gemini_data_dir / "train.tsv", sep="\t", index=False, header=False, quoting=csv.QUOTE_NONE)
dev_updated_df.to_csv(gemini_data_dir / "dev.tsv",     sep="\t", index=False, header=False, quoting=csv.QUOTE_NONE)
test_updated_df.to_csv(gemini_data_dir / "test.tsv",    sep="\t", index=False, header=False, quoting=csv.QUOTE_NONE)

In [9]:
# Load in dataframes

# Define path
gemini_data_dir = Path(
    r"D:\Users\Jessica\kg-construction\data\00_raw\biorex\ncbi_relation_gemini_annotations\processed"
)

# Define columns in exact original order
columns = [
    "pmid",
    "Entity_1_Type",
    "Entity_2_Type",
    "Entity_1_ID",
    "Entity_2_ID",
    "is_in_same_sent",
    "min_sents_window",
    "sentence",
    "extra",
    "Relation_Type",
]

# Read files back into DataFrames
train_updated_df = pd.read_csv(
    gemini_data_dir / "train.tsv",
    sep="\t",
    header=None,
    names=columns,
    quoting=csv.QUOTE_NONE,
)
dev_updated_df = pd.read_csv(
    gemini_data_dir / "dev.tsv",
    sep="\t",
    header=None,
    names=columns,
    quoting=csv.QUOTE_NONE,
)
test_updated_df = pd.read_csv(
    gemini_data_dir / "test.tsv",
    sep="\t",
    header=None,
    names=columns,
    quoting=csv.QUOTE_NONE,
)

In [10]:
# 1. Combine entity columns from all three DataFrames
combined = pd.concat([train_updated_df, dev_updated_df, test_updated_df])[
    ['Entity_1_Type', 'Entity_2_Type']
].dropna()

# 2. Sort pairs alphabetically row-by-row to ignore order (e.g., ['B', 'A'] -> ['A', 'B'])
sorted_pairs = np.sort(combined.values, axis=1)

# 3. Extract unique pairs as a DataFrame
unique_pairs_df = (
    pd.DataFrame(sorted_pairs, columns=['Entity_A', 'Entity_B'])
    .drop_duplicates()
    .reset_index(drop=True)
)

print(unique_pairs_df)

                     Entity_A                    Entity_B
0           GeneOrGeneProduct           GeneOrGeneProduct
1              ChemicalEntity           GeneOrGeneProduct
2  DiseaseOrPhenotypicFeature           GeneOrGeneProduct
3              ChemicalEntity  DiseaseOrPhenotypicFeature
4              ChemicalEntity              ChemicalEntity


### Compare train, test, dev, datasets between Gemini and Original BioRED

In [185]:
train_updated_df

,pmid,Entity_1_Type,Entity_2_Type,Entity_1_ID,Entity_2_ID,is_in_same_sent,min_sents_window,sentence,extra,Relation_Type
0,10491763,GeneOrGeneProduct,GeneOrGeneProduct,6927,rs74805019,True,0,What is [Litcoin] between @GeneOrGeneProductSr...,,None
1,10491763,GeneOrGeneProduct,GeneOrGeneProduct,3630,rs74805019,True,0,What is [Litcoin] between @GeneOrGeneProductSr...,,None
2,10491763,ChemicalEntity,GeneOrGeneProduct,D005947,6927,True,0,What is [Litcoin] between @ChemicalEntitySrc$ ...,,None
3,10491763,GeneOrGeneProduct,GeneOrGeneProduct,3651,rs74805019,True,0,What is [Litcoin] between @GeneOrGeneProductSr...,,None
4,10491763,ChemicalEntity,GeneOrGeneProduct,D005947,3172,True,0,What is [Litcoin] between @ChemicalEntitySrc$ ...,,None
...,...,...,...,...,...,...,...,...,...,...
22891,24927617,ChemicalEntity,DiseaseOrPhenotypicFeature,D019821,D012206,True,0,What is [Litcoin] between @ChemicalEntitySrc$ ...,,Positive_Correlation
22892,24927617,ChemicalEntity,ChemicalEntity,D012254,C417083,True,0,What is [Litcoin] between @ChemicalEntitySrc$ ...,,Cotreatment
22893,24927617,ChemicalEntity,ChemicalEntity,D019821,D000998,True,0,What is [Litcoin] between @ChemicalEntitySrc$ ...,,None
22894,24927617,ChemicalEntity,ChemicalEntity,C486464,D012254,True,0,What is [Litcoin] between @ChemicalEntitySrc$ ...,,Cotreatment


In [186]:
import pandas as pd


def compare_relation_counts(df_orig, df_updated, split_name="train"):
    """Normalizes NaN/None values to 'None' and compares Relation_Type value counts."""
    # Standardize NaN, None, and string 'None' into a uniform 'None' string
    s_orig = (
        df_orig["Relation_Type"]
        .fillna("None")
        .replace({"nan": "None", None: "None"})
        .astype(str)
    )
    s_upd = (
        df_updated["Relation_Type"]
        .fillna("None")
        .replace({"nan": "None", None: "None"})
        .astype(str)
    )

    # Get value counts including 'None'
    vc_orig = s_orig.value_counts(dropna=False)
    vc_upd = s_upd.value_counts(dropna=False)

    # Combine into a single comparison DataFrame
    comp_df = (
        pd.DataFrame(
            {f"{split_name}_original": vc_orig, f"{split_name}_updated": vc_upd}
        )
        .fillna(0)
        .astype(int)
    )

    # Calculate difference and check for exact match
    comp_df["Diff"] = (
        comp_df[f"{split_name}_updated"] - comp_df[f"{split_name}_original"]
    )
    comp_df["Match"] = comp_df["Diff"] == 0

    return comp_df


# 1. Define your paired splits
splits = {
    "Train": (train_df, train_updated_df),
    "Dev": (dev_df, dev_updated_df),
    "Test": (test_df, test_updated_df),
}

# 2. Run and print comparisons for all splits
all_matches = True

for split_name, (df_orig, df_upd) in splits.items():
    print(f"\n=================== {split_name.upper()} SPLIT ===================")
    comparison_df = compare_relation_counts(
        df_orig, df_upd, split_name=split_name.lower()
    )
    print(comparison_df)

    # Check if all counts match
    split_matches = comparison_df["Match"].all()
    if not split_matches:
        all_matches = False
        print(
            f"\n⚠️ Discrepancy found in {split_name} split! Check rows where Match == False."
        )

if all_matches:
    print(
        "\n✅ Success: All Relation_Type value counts match perfectly across all splits!"
    )


=================== TRAIN SPLIT ===================
                      train_original  train_updated  Diff  Match
Relation_Type                                                   
Association                     2184            864 -1320  False
Bind                              60             51    -9  False
Comparison                        28             31     3  False
Conversion                         3             19    16  False
Cotreatment                       31             33     2  False
Drug_Interaction                  11             11     0   True
Negative_Correlation             763            963   200  False
None                           18696          19485   789  False
Positive_Correlation            1088           1407   319  False

⚠️ Discrepancy found in Train split! Check rows where Match == False.

=================== DEV SPLIT ===================
                      dev_original  dev_updated  Diff  Match
Relation_Type                                    